# GoEmotions Multi-Label Classification with RoBERTa

Clean Trainer-based implementation for easy hyperparameter tuning.

Task: Multi-label emotion classification.


In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset


## Load Dataset

In [2]:
df = pd.read_csv('goemotions_cleaned.csv')

# This cleaned file includes metadata columns. Keep only text + label columns.
# Label columns start at 'admiration' and are 0/1 indicators.
label_start = df.columns.get_loc('admiration')
emotion_cols = list(df.columns[label_start:])
df = df[['text'] + emotion_cols]

print('Rows:', len(df))
print('Number of emotions:', len(emotion_cols))


Rows: 58004
Number of emotions: 28


## Train / Validation / Test Split
-  80/10/10

In [3]:
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

print(f"Train split rows: {len(df_train)} ({len(df_train)/len(df)*100:.2f}%)")
print(f"Validation split rows: {len(df_val)} ({len(df_val)/len(df)*100:.2f}%)")
print(f"Test split rows: {len(df_test)} ({len(df_test)/len(df)*100:.2f}%)")

train_texts = df_train['text'].tolist()
val_texts = df_val['text'].tolist()
test_texts = df_test['text'].tolist()

y_train = df_train[emotion_cols].values
y_val = df_val[emotion_cols].values
y_test = df_test[emotion_cols].values


Train split rows: 46403 (80.00%)
Validation split rows: 5800 (10.00%)
Test split rows: 5801 (10.00%)


## Tokenization

In [4]:
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)


## Dataset Class (Trainer Compatible)

In [5]:
class GoEmotionsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = GoEmotionsDataset(train_encodings, y_train)
val_dataset = GoEmotionsDataset(val_encodings, y_val)
test_dataset = GoEmotionsDataset(test_encodings, y_test)


## Load Model

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=len(emotion_cols)
)

model.config.problem_type = 'multi_label_classification'


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Metrics

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0)
    }


## Training Configuration

In [8]:
%pip install -U accelerate

Note: you may need to restart the kernel to use updated packages.


In [ ]:
training_args = TrainingArguments(
    output_dir='./goemotions_roberta',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    logging_steps=100,
    warmup_steps=200,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


- The only thing you need to change between runs is the run_name variable at the top — just update it to something descriptive like "run2_epochs5_lr2e5" before each new training run. The comparison section at the bottom will automatically pick up all saved CSVs and print them side by side.

In [ ]:
# Extract and display validation history
log_history = trainer.state.log_history
eval_logs = [log for log in log_history if 'eval_f1_micro' in log]
results_df = pd.DataFrame(eval_logs)[['epoch', 'eval_loss', 'eval_f1_micro', 'eval_f1_macro']]
print(results_df)

# Save to CSV with descriptive name — change this name each run!
run_name = "run1_epochs3_lr2e5_bs32_4epochs"
results_df.to_csv(f"{run_name}.csv", index=False)
print(f"\nSaved to {run_name}.csv")

# Step 8 - test with the test dataset

- only proceed when we're ready to test

### won't training again with different number of epochs be bad for the model since the model has already 'seen' the test set?

- yes:  if you look at your test score, tweak something (epochs, learning rate, etc.), retrain, and check test again repeatedly. Each time you peek at the test score and make a decision based on it, you're indirectly "leaking" information from the test set into your choices.

- no: The model never trained on it — it was only used in trainer.evaluate(test_dataset) for read-only scoring, not for updating the model's weights. So it's still a valid final benchmark.

- best practice
1. train 
2. check EVAL score 
3. adjust hyperparameters (only use this to make decisions)
4. retrain

a. final test score -> run ONCE at the very end when you're done experimenting

In [ ]:
results = trainer.evaluate(test_dataset)
print("Final Test Results:")
print(f"  F1 Micro: {results['eval_f1_micro']:.4f}")
print(f"  F1 Macro: {results['eval_f1_macro']:.4f}")

In [ ]:
def predict_emotions(text, threshold=0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.sigmoid(outputs.logits[0])
    results = {emotion_cols[i]: round(probs[i].item(), 3) for i in range(len(emotion_cols))}
    # Only show emotions above the threshold
    active = {k: v for k, v in results.items() if v >= threshold}
    return dict(sorted(active.items(), key=lambda x: -x[1]))

print(predict_emotions("I can't believe how amazing that was!"))
# → {'admiration': 0.82, 'excitement': 0.71, 'joy': 0.63}

print(predict_emotions("This makes me so angry and disappointed."))
# → {'anger': 0.79, 'disappointment': 0.68, 'annoyance': 0.54}
